<a href="https://colab.research.google.com/github/peperjet/deep-learning/blob/main/Subword_Tokenizer/13_2_naver_movie_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

센텐스피스(SentencePiece) 네이버 영화리뷰 토큰화하기


In [1]:
import pandas as pd
import sentencepiece as spm
import urllib.request
import csv


In [2]:
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings.txt", filename="ratings.txt")


('ratings.txt', <http.client.HTTPMessage at 0x7ac11e715130>)

In [4]:
naver_df = pd.read_table('/content/ratings.txt')
naver_df[:5]


,id,document,label
0,8112052,어릴때보고 지금다시봐도 재밌어요ㅋㅋ,1
1,8132799,"디자인을 배우는 학생으로, 외국디자이너와 그들이 일군 전통을 통해 발전해가는 문화산...",1
2,4655635,폴리스스토리 시리즈는 1부터 뉴까지 버릴께 하나도 없음.. 최고.,1
3,9251303,와.. 연기가 진짜 개쩔구나.. 지루할거라고 생각했는데 몰입해서 봤다.. 그래 이런...,1
4,10067386,안개 자욱한 밤하늘에 떠 있는 초승달 같은 영화.,1


총 20만개의 샘플 존재

In [5]:
print('리뷰 개수 :',len(naver_df)) # 리뷰 개수 출력

리뷰 개수 : 200000


네이버 영화 리뷰 데이터의 경우 Null 값이 존재하므로 이를 제거한 후 수행

In [6]:
print(naver_df.isnull().values.any())


True


In [7]:
naver_df = naver_df.dropna(how = 'any') # Null 값이 존재하는 행 제거
print(naver_df.isnull().values.any()) # Null 값이 존재하는지 확인


False


In [8]:
print('리뷰 개수 :',len(naver_df)) # 리뷰 개수 출력


리뷰 개수 : 199992


최종적으로 199,992개의 샘플을 naver_review.txt 파일에 저장한 후에 센텐스피스를 통해 단어 집합을 생성합니다.

In [9]:
with open('naver_review.txt', 'w', encoding='utf8') as f:
    f.write('\n'.join(naver_df['document']))


In [10]:
spm.SentencePieceTrainer.Train('--input=naver_review.txt --model_prefix=naver --vocab_size=5000 --model_type=bpe --max_sentence_length=9999')


vocab 생성이 완료되면 naver.model, naver.vocab 파일 두개가 생성 됩니다. .vocab 에서 학습된 subwords를 확인할 수 있습니다.

In [11]:
vocab_list = pd.read_csv('naver.vocab', sep='\t', header=None, quoting=csv.QUOTE_NONE)
vocab_list[:10]


,0,1
0,<unk>,0
1,<s>,0
2,</s>,0
3,..,0
4,영화,-1
5,▁영화,-2
6,▁이,-3
7,▁아,-4
8,...,-5
9,ᄏᄏ,-6


Vocabulary 에는 unknown, 문장의 시작, 문장의 끝을 의미하는 special token이 0, 1, 2에 사용되었습니다.

In [12]:
len(vocab_list)


5000

In [13]:
sp = spm.SentencePieceProcessor()
vocab_file = "naver.model"
sp.load(vocab_file)


True

In [14]:
lines = [
  "뭐 이딴 것도 영화냐.",
  "진짜 최고의 영화입니다 ㅋㅋ",
]
for line in lines:
  print(line)
  print(sp.encode_as_pieces(line))
  print(sp.encode_as_ids(line))
  print()


뭐 이딴 것도 영화냐.
['▁뭐', '▁이딴', '▁것도', '▁영화냐', '.']
[136, 970, 1299, 2593, 3276]

진짜 최고의 영화입니다 ㅋㅋ
['▁진짜', '▁최고의', '▁영화입니다', '▁ᄏᄏ']
[54, 204, 825, 121]



In [19]:
sp.GetPieceSize() # 단어 집합의 크기를 확인

5000

In [20]:
sp.IdToPiece(4) # 정수로부터 맵핑되는 서브워드 변환


'영화'

In [21]:
sp.PieceToId('영화') # 서브워드로부터 맵핑되는 정수  변환


4

In [23]:
sp.DecodeIds([54, 200, 821, 85]) # 정수 시퀀스로부터 문장으로 변환


'진짜 원 산~~'

In [25]:
sp.DecodePieces(['▁진짜', '▁최고의', '▁영화입니다', '▁ᄏᄏ']) # 서브워드 시퀀스로부터


'진짜 최고의 영화입니다 ᄏᄏ'

In [27]:
# encode : 문장으로부터 인자값에 따라서 정수 시퀀스 또는 서브워드 싀퀀스로 변환 가능
print(sp.encode('진짜 최고의 영화입니다 ㅋㅋ', out_type=str))
print(sp.encode('진짜 최고의 영화입니다 ㅋㅋ', out_type=int))


['▁진짜', '▁최고의', '▁영화입니다', '▁ᄏᄏ']
[54, 204, 825, 121]
